# 2 · Connect

Before downloading any imagery, confirm that Planetary Computer is reachable from your machine.

Planetary Computer is an open data catalogue — no account or login is needed for Landsat and
Sentinel-2. What you *do* need is the ability to sign asset URLs: a short-lived access token
that lets your machine read the actual image files. This notebook checks both things.

Run the two cells in order. If both print `OK`, move on to [3 · Tides](03_tides.ipynb).

## Step 1 - Catalogue search

A STAC search returns scene *metadata* — footprints, timestamps, cloud cover — without
downloading any pixels. This step proves that your machine can reach the Planetary
Computer API and that queries work.

In [1]:
import pystac_client
import planetary_computer

# Open the Planetary Computer STAC catalogue.
# sign_inplace automatically adds access tokens to any asset URL we retrieve.
PC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
catalog = pystac_client.Client.open(PC_URL, modifier=planetary_computer.sign_inplace)

# Run a small test search — Landsat scenes over the Oosterschelde, Q1 2023.
# Change bbox and datetime to your own area later; this is just a connectivity check.
search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=(3.96, 51.55, 4.10, 51.62),   # (west, south, east, north) in decimal degrees
    datetime="2023-01-01/2023-03-31",
    query={"eo:cloud_cover": {"lt": 80}},
)

# item_collection() executes the search and returns the matched scene metadata.
items = search.item_collection()
print(f"Connection OK — found {len(items)} Landsat scenes in the test window.")

Connection OK — found 12 Landsat scenes in the test window.


> **Zero scenes is not a failure.** It means the query found nothing matching your
> filters — try a wider `bbox`, a longer `datetime` range, or a higher cloud-cover limit.
> Only a Python exception or network error means the connection itself is broken.

## Step 2 - Asset signing

Scene metadata is public; the actual image files are protected by short-lived tokens.
`planetary_computer.sign()` fetches those tokens and attaches them to the asset URLs —
this is what replaces a traditional account login. This step confirms that signing works
and that a real file URL is returned.

In [2]:
if len(items) > 0:
    # Sign the first scene — this fetches a short-lived SAS token from Microsoft
    # and attaches it to every asset URL in the item.
    signed = planetary_computer.sign(items[0])

    # 'red' is the red band (B4 for Landsat 8/9). We strip the token from the URL
    # before printing so the output stays readable.
    href = signed.assets["red"].href
    print("Signing OK — assets are readable.")
    print("Signed URL (token stripped):", href.split("?")[0])
else:
    print("No scenes to sign — widen the search above and re-run.")

Signing OK — assets are readable.
Signed URL (token stripped): https://landsateuwest.blob.core.windows.net/landsat-c2/level-2/standard/oli-tirs/2023/199/024/LC08_L2SP_199024_20230325_20230404_02_T1/LC08_L2SP_199024_20230325_20230404_02_T1_SR_B4.TIF


Both cells printed `OK`? You are ready.

**Next: [3 · Tides →](03_tides.ipynb)**

---

:::{dropdown} Troubleshooting

**Network or TLS error** — check that `https://planetarycomputer.microsoft.com` is
reachable from your network. Corporate proxies sometimes block it.

**`0` scenes returned** — widen `bbox`, extend `datetime`, or raise the cloud-cover limit.
Zero results is a query issue, not a connection failure.

**Import errors** — make sure you started Jupyter with `uv run jupyter lab` from the
repository root, not a system-wide Jupyter. See [1 · Setup](01_setup.md).
:::